In [ ]:
using RhapsodieDirect
using Statistics
include("src/noise_validation.jl")
CorrelatedNoise(2.0, 1.0, 128)

In [ ]:
# Ajoute ces fonctions avant validate_noise_model2
function compute_variance(matrix::AbstractMatrix)
    return var(matrix)
end

function theoretical_variance(model::CorrelatedNoise)
    return model.A / (4π * model.σ²)
end


function validate_noise_model2(model::CorrelatedNoise; σ²_values=logrange(0.1, 50, 30), n_samples=100)
    # Check if Plots is available
    if !isdefined(Main, :Plots)
        try
            @eval Main using Plots
        catch
            @warn "Plots.jl not available. Install with: using Pkg; Pkg.add(\"Plots\")"
            return
        end
    end
    
    println("🔍 VALIDATING CORRELATED NOISE MODEL")
    println("="^50)
    
    # Generate test ranges
    empirical_vars = Float64[]
    theoretical_vars = Float64[]
    
    for σ² in σ²_values
        # Create temporary model with different σ²
        temp_model = CorrelatedNoise(model.A, σ², model.N)
        
        # Generate samples and compute empirical variance
        variances = [compute_variance(generate_correlated_noise(temp_model)) for _ in 1:n_samples]
        push!(empirical_vars, mean(variances))
        
        # Theoretical variance
        push!(theoretical_vars, theoretical_variance(temp_model))
    end
    
    # Plot results
    p = Main.Plots.plot(σ²_values, theoretical_vars, 
                       label="Theoretical", linewidth=2, 
                       xlabel="σ²", ylabel="Variance",
                       title="Noise Model Validation")
    Main.Plots.scatter!(p, σ²_values, empirical_vars, 
                       label="Empirical", markersize=4)
    
    # Display results
    println("✅ Validation completed")
    println("   Max relative error: $(maximum(abs.(empirical_vars .- theoretical_vars) ./ theoretical_vars) * 100)%")
    
    return p
end

In [ ]:
# include("src/noise_validation.jl")
import RhapsodieDirect: validate_noise_model
validate_noise_model2(CorrelatedNoise(2.0, 1.0, 128))

# Équations LaTeX - Génération de Bruit Corrélé

## 1. Modèle de Densité Spectrale de Puissance

$$
P(\vec{k}) = A \cdot \exp\left(-\sigma^2 |\vec{k}|^2\right)
$$

où :
- $\vec{k} = (k_x, k_y)$ est le vecteur d'onde dans l'espace de Fourier
- $|\vec{k}|^2 = k_x^2 + k_y^2$ est la norme au carré du vecteur d'onde
- $A > 0$ est le paramètre d'amplitude contrôlant la variance totale du bruit
- $\sigma^2 > 0$ est le paramètre de largeur spectrale contrôlant les corrélations spatiales

## 2. Discrétisation sur Grille

$$
P[i,j] = A \cdot \exp\left(-\sigma^2 (2\pi)^2 \left(\nu_i^2 + \nu_j^2\right)\right)
$$

avec les fréquences discrètes :
$$
\nu_i = \frac{i - 1}{N} - \frac{1}{2} \quad \text{pour} \quad i \in \{1, 2, \ldots, N\}
$$

## 3. Génération du Bruit Corrélé

Le processus de génération se décompose en trois étapes :

### Étape 1 : Bruit Blanc Gaussien
$$
\xi[i,j] \sim \mathcal{N}(0, 1) \quad \text{i.i.d.}
$$

### Étape 2 : Filtrage dans l'Espace de Fourier
$$
\tilde{\varepsilon}[\vec{k}] = \mathcal{F}\{\xi\}[\vec{k}] \cdot \sqrt{P[\vec{k}]}
$$

### Étape 3 : Transformation Inverse
$$
\varepsilon[i,j] = \text{Re}\left\{\mathcal{F}^{-1}\{\tilde{\varepsilon}\}[i,j]\right\}
$$

## 4. Propriétés Statistiques

### Variance Théorique
$$
\text{Var}(\varepsilon) = \frac{A}{4\pi\sigma^2}
$$

### Matrice de Covariance
$$
C = \langle \varepsilon \varepsilon^T \rangle = \mathcal{F}^{-1} \text{diag}(P(\vec{k})) \mathcal{F}
$$

### Matrice de Précision
$$
W = C^{-1} = \mathcal{F}^{-1} \text{diag}\left(\frac{1}{P(\vec{k}) + \epsilon}\right) \mathcal{F}
$$

où $\epsilon > 0$ est un terme de régularisation pour éviter la division par zéro.

## 5. Génération des Données Observées

### Modèle de Formation des Données
$$
y = \mathsf{A}(\tilde{x}) + \varepsilon
$$

où :
- $y$ représente les données observées
- $\mathsf{A}$ est l'opérateur de formation d'image (convolution, transformations géométriques)
- $\tilde{x} = x + \lambda s$ est l'objet total (disque + contamination stellaire)
- $\varepsilon$ est le bruit corrélé généré selon le processus ci-dessus (3.)

\begin{align}
\xi[i,j] &\sim \mathcal{N}(0,1) \quad \text{(bruit blanc)} \\
\tilde{\xi}[\vec{k}] &= \mathcal{F}\{\xi\}[\vec{k}] \quad \text{(FFT)} \\
\tilde{\varepsilon}[\vec{k}] &= \tilde{\xi}[\vec{k}] \cdot \sqrt{P[\vec{k}]} \quad \text{(filtrage)} \\
\varepsilon[i,j] &= \text{Re}\{\mathcal{F}^{-1}\{\tilde{\varepsilon}\}\}[i,j] \quad \text{(IFFT)}
\end{align}

### Vraisemblance
$$
\log p(y|x,\lambda) = -\frac{1}{2}(y - \mathsf{A}(\tilde{x}))^T W (y - \mathsf{A}(\tilde{x})) + \text{cste}
$$

### Gradient
$$
\nabla_x \log p(y|x,\lambda) = \mathsf{A}^T W (\mathsf{A}(\tilde{x}) - y)
$$